# Prática 1 — Da força bruta ao privilégio

Esta prática refaz, com `pandas`, a caçada que a Seção~1.5 do capítulo descreve no Kibana.
Cada passo mostra primeiro a **consulta ingênua**, que cai na isca, e depois a **pergunta refinada**,
que responde à hipótese. No fim, as respostas são conferidas contra o gabarito que o gerador do
cenário recalcula por código.

**Hipótese.** Se alguém adivinhou uma senha da VPN, a mesma origem externa acumula falhas em muitas contas diferentes e, em algum momento, uma sessão bem-sucedida.

## Preparar o ambiente

Rode esta célula primeiro. Na nuvem ela baixa o material; na sua máquina ela reconhece que já está tudo no lugar.


In [ ]:
# Em nuvem, o caderno roda sozinho: este passo busca o apoio e a evidência uma vez.
# Na sua máquina, com o repositório do minicurso em volta, ele não faz nada.
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

MATERIAL = "https://hikari-edu.github.io/minicurso"

if not Path("eventos.py").exists() and not (Path.cwd() / "eventos.py").exists():
    urlretrieve(f"{MATERIAL}/notebooks/eventos.py", "eventos.py")

evidencia = Path("cenario/saida/aurora-telecom.json")
if not evidencia.exists() and not (Path.cwd().parent / "cenario" / "saida" / "aurora-telecom.json").exists():
    evidencia.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(f"{MATERIAL}/cenario/aurora-evidencia.zip", "aurora-evidencia.zip")
    with ZipFile("aurora-evidencia.zip") as pacote:
        pacote.extractall(evidencia.parent)
    print("evidência baixada para", evidencia.parent.resolve())
else:
    print("evidência já disponível")


## Carregar a evidência

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
import pandas as pd

from eventos import conferir, ler_eventos, ler_gabarito

eventos = ler_eventos()
gabarito = ler_gabarito()
print(f"{len(eventos)} eventos, de {eventos['@timestamp'].min()} a {eventos['@timestamp'].max()}")
eventos["event.dataset"].value_counts()

49277 eventos, de 2026-03-09 00:00:07+00:00 a 2026-03-11 23:59:42+00:00


event.dataset
dns           22159
firewall       9655
fileserver     7340
edr            4696
vpn            2967
auth           2460
Name: count, dtype: int64

## Passo 1 — o volume de falhas, e por que ele engana

A primeira pergunta de quase todo mundo é *qual origem falhou mais*. No Kibana:

```
event.dataset:"vpn" and event.outcome:"failure"  →  contagem por source.ip
```

In [2]:
falhas = eventos[(eventos["event.dataset"] == "vpn") & (eventos["event.outcome"] == "failure")]
falhas["source.ip"].value_counts().head(5)

source.ip
10.20.5.14        1440
203.0.113.77      1026
177.39.199.144       2
177.23.183.32        1
177.37.197.130       1
Name: count, dtype: int64

A origem do topo é uma sonda de monitoramento com senha vencida: muitas falhas, **uma conta só**, e ela nunca entra. É a isca do caso.

In [3]:
falhas.groupby("source.ip")["user.name"].nunique().sort_values(ascending=False).head(5)

source.ip
203.0.113.77      62
177.39.199.144     2
10.20.5.14         1
177.20.180.11      1
177.23.183.32      1
Name: user.name, dtype: int64

## Passo 2 — a pergunta refinada: contas distintas, entre quem entrou

```
event.dataset:"vpn" and event.outcome:"failure"  →  source.ip por contagem única de user.name
```

In [4]:
vpn = eventos[eventos["event.dataset"] == "vpn"]
com_sucesso = set(vpn[vpn["event.outcome"] == "success"]["source.ip"])
contas_por_origem = falhas.groupby("source.ip")["user.name"].nunique()
candidatas = contas_por_origem[contas_por_origem.index.isin(com_sucesso)].sort_values(ascending=False)
origem = candidatas.index[0]
contas = int(candidatas.iloc[0])
print(origem, contas)
candidatas.head(5)

203.0.113.77 62


source.ip
203.0.113.77      62
177.39.199.144     2
177.20.180.11      1
177.23.183.32      1
177.24.184.39      1
Name: user.name, dtype: int64

## Passo 3 — a conta que cedeu e o minuto zero

In [5]:
sessao = vpn[(vpn["source.ip"] == origem) & (vpn["event.outcome"] == "success")].iloc[0]
conta = sessao["user.name"]
minuto_zero = sessao["@timestamp"].strftime("%Y-%m-%dT%H:%M:%S.000Z")
endereco_interno = sessao["client.nat.ip"]
print(conta, minuto_zero, endereco_interno)

c.moura 2026-03-10T05:17:43.000Z 10.8.200.23


## Passo 4 — o pivô de endereço

Filtrar pela conta misturaria o ataque com o expediente: `c.moura` tem logons legítimos. O pivô certo é o endereço interno que só o atacante usou.

In [6]:
auth = eventos[eventos["event.dataset"] == "auth"]
remotos = auth[(auth["source.ip"] == endereco_interno) & (auth["winlog.logon.type"] == "10")]
servidor = remotos.iloc[0]["host.name"]
remotos[["@timestamp", "user.name", "host.name", "event.action"]].head()

,@timestamp,user.name,host.name,event.action
17422,2026-03-10 05:24:10+00:00,c.moura,SRV-FIN-02,Logon remoto concluído


## Passo 5 — a persistência

Cuidado com o campo: nos eventos de gestão de contas, `user.name` é **quem agiu** (a conta roubada) e
`target.user.name` é **a conta criada**. Ler o campo errado aqui devolve `c.moura` e esconde o que o
adversário deixou para trás.

In [7]:
gestao = auth[(auth["host.name"] == servidor) & (auth["event.code"].isin(["4720", "4732"]))]
# Em 4720 e 4732, user.name é quem agiu e target.user.name é a conta afetada.
nova_conta = gestao.iloc[0]["target.user.name"]
gestao[["@timestamp", "user.name", "target.user.name", "group.name", "event.code", "event.action"]]

,@timestamp,user.name,target.user.name,group.name,event.code,event.action
17444,2026-03-10 05:30:48+00:00,c.moura,svc_relatorios,NaN,4720,Conta de usuário criada
17449,2026-03-10 05:31:35+00:00,c.moura,svc_relatorios,Administradores,4732,Membro adicionado a grupo local


## Conferência contra o gabarito

As respostas do notebook precisam bater com as que o gerador recalcula por código sobre todos os eventos.

In [8]:
for chave, obtido in [("c1-origem", origem), ("c1-contas", contas), ("c1-conta", conta),
                      ("c1-instante", minuto_zero), ("c1-servidor", servidor),
                      ("c1-persistencia", nova_conta)]:
    print(conferir(chave, obtido, gabarito))

ok   c1-origem: obtido '203.0.113.77', gabarito '203.0.113.77'
ok   c1-contas: obtido 62, gabarito '62'
ok   c1-conta: obtido 'c.moura', gabarito 'c.moura'
ok   c1-instante: obtido '2026-03-10T05:17:43.000Z', gabarito '2026-03-10T05:17:43.000Z'
ok   c1-servidor: obtido 'SRV-FIN-02', gabarito 'SRV-FIN-02'
ok   c1-persistencia: obtido 'svc_relatorios', gabarito 'svc_relatorios'


## Exercício

Escreva a consulta que encontraria o mesmo ataque se o adversário tivesse limitado as tentativas a cinco contas. Que outro sinal passaria a ser necessário?